In [1]:
import pandas as pd 
import torch
from torch import nn
from matplotlib import pyplot as plt 

In [2]:
df = pd.read_csv("./loan_data.csv")
df = df[["person_income", "loan_intent", "loan_percent_income", "previous_loan_defaults_on_file", "loan_status"]]
print(df.head())
df.dtypes
df.shape

   person_income loan_intent  loan_percent_income  \
0        71948.0    PERSONAL                 0.49   
1        12282.0   EDUCATION                 0.08   
2        12438.0     MEDICAL                 0.44   
3        79753.0     MEDICAL                 0.44   
4        66135.0     MEDICAL                 0.53   

  previous_loan_defaults_on_file  loan_status  
0                             No            1  
1                            Yes            0  
2                             No            1  
3                             No            1  
4                             No            1  


(45000, 5)

In [3]:
df["previous_loan_defaults_on_file"] = df["previous_loan_defaults_on_file"]=="Yes"
df.head()
df["previous_loan_defaults_on_file"] = df["previous_loan_defaults_on_file"].astype('float32')
df.head()
df = pd.get_dummies(df, columns=["loan_intent"]).astype("float32")
df.head()

,person_income,loan_percent_income,previous_loan_defaults_on_file,loan_status,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,71948.0,0.49,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,12282.0,0.08,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,12438.0,0.44,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,79753.0,0.44,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,66135.0,0.53,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [4]:
df_train = df.sample(frac=0.8, random_state=50)
rest = df.drop(df_train.index)
df_val = rest.sample(frac = 0.5, random_state =50)
df_test = rest.drop(df_val.index)
print(df_train.shape, df_val.shape, df_test.shape)

(36000, 10) (4500, 10) (4500, 10)


In [5]:
# Normalization of data
x_train = torch.tensor(df_train.drop(columns="loan_status").values, dtype = torch.float32)
x_train_mean = x_train.mean(dim=0, keepdim=True)
x_train_std = x_train.std(dim=0, keepdim=True)
x_train = (x_train - x_train_mean)/x_train_std
y_train = torch.tensor(df_train["loan_status"].values, dtype=torch.float32).view(-1,1)
x_val = torch.tensor(df_val.drop(columns=["loan_status"]).values, dtype=torch.float32)
x_val_mean = x_val.mean(dim=0, keepdim=True)
x_val_std = x_val.std(dim=0, keepdim=True)
x_val = (x_val - x_val_mean)/x_val_std
y_val = torch.tensor(df_val["loan_status"].values, dtype=torch.float32).view(-1,1)



In [6]:
## Model setup
model = nn.Sequential(
    nn.Linear(in_features=9,out_features= 32),
    nn.ReLU(),
    nn.Linear(32,16),
    nn.ReLU(),
    #nn.Linear(16,8),
    #nn.ReLU(),
    nn.Linear(8,1)
)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


In [7]:
##Train the model
def train_model(model, loss_fn, optimizer, df_train, df_val, epochs=10000):
    train_losses =[]
    val_losses = []
    for i in range(epochs):
        model.train()
        optimizer.zero_grad()
        y_pred_train = model(x_train)
        loss_train = loss_fn(y_pred_train, y_train)
        loss_train.backward()
        optimizer.step()
        train_losses.append(loss_train.item())

        model.eval()
        with torch.no_grad():
            y_pred_val = model(x_val)
            loss_val = loss_fn(y_pred_val, y_val)
            val_losses.append(loss_val.item())

    if i%1000==0:
        print("Training loss : ", loss_train.item(), " and validation loss is", loss_val.item())        

    return train_losses, val_losses

train_model(model, loss_fn, optimizer, df_train, df_val)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (36000x16 and 8x1)

In [ ]:
## Evalute the model on test data
x_test = torch.tensor(df_test.drop(columns=["loan_status"]).values, dftype=torch.float32)
x_test = (x_test-x_train_mean)/x_train_std
y_test = torch.tensor(df_test["loan_status"].values, dftype=torch.float32).view(-1,1)
model.eval()
with torch.no_grad():
    y_pred_test = model(x_test)
    loss_test = loss_fn(y_pred_test, y_test)
    print("Test loss: ", loss_test)
